# Phase 3 — Model Development  
## Visit Risk Classification Model (02_risk_model.ipynb)

### Business Objective
The goal of this model is to predict the **risk level of a hospital visit**  
(`Low`, `Medium`, `High`) in order to:

- Help hospital administrators prioritize high-risk cases  
- Support better clinical resource allocation  
- Improve operational planning  

### Target Variable
`risk_score`

This is a **multi-class classification problem**.

## Step 1 — Import Required Libraries

In [29]:

# Basic data handling
import pandas as pd
import numpy as np

# Machine Learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Model saving
import joblib

from sklearn.preprocessing import StandardScaler


## Step 2 — Load Modeling Dataset

This dataset was generated in Phase 2 after performing:
- Data cleaning
- Feature engineering
- SQL-consistent joins

File used: `model_table.csv`

In [30]:

# Load dataset
df = pd.read_csv("model_table.csv")

# Check dataset shape
print("Dataset Shape:", df.shape)

# Preview first 5 rows
df.head()
print(df["risk_score"].value_counts(normalize=True))

Dataset Shape: (25000, 26)
risk_score
Low       0.49880
Medium    0.29984
High      0.20136
Name: proportion, dtype: float64



## Step 3 — Convert Date Columns

We convert `visit_date` to datetime format  
so that we can perform a proper time-based train/test split.

In [31]:
df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")

## Step 4 — Define Target Variable

Target: `risk_score`

In [32]:

# Define target variable
y = df["risk_score"]

# Check class distribution
print("Risk Score Distribution:")
print(y.value_counts())


Risk Score Distribution:
risk_score
Low       12470
Medium     7496
High       5034
Name: count, dtype: int64



## Step 5 — Feature Selection

We select clinically and operationally relevant features.

In [33]:

# Convert categorical variables using one-hot encoding
df_encoded = pd.get_dummies(
    df,
    columns=["department", "visit_type"],
    drop_first=True
)

# Base numeric features
feature_columns = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "days_since_registration",
    "visit_month",
    "visit_dayofweek"
]

# Add encoded categorical columns
encoded_columns = [
    col for col in df_encoded.columns
    if col.startswith("department_") or col.startswith("visit_type_")
]

feature_columns.extend(encoded_columns)


## Step 6 — Time-Based Train/Test Split

As required,

- Earliest 80% of visits → Training  
- Latest 20% of visits → Testing  

This prevents data leakage and simulates real-world deployment.

In [34]:

# Sort dataset by visit_date
df_sorted = df_encoded.sort_values("visit_date")

X_sorted = df_sorted[feature_columns].fillna(0)
y_sorted = df_sorted["risk_score"]

# Create split index
split_index = int(len(df_sorted) * 0.8)

# Training data
X_train = X_sorted.iloc[:split_index]
y_train = y_sorted.iloc[:split_index]

# Testing data
X_test = X_sorted.iloc[split_index:]
y_test = y_sorted.iloc[split_index:]

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)


Training Size: (20000, 16)
Testing Size: (5000, 16)


## Step 7 — Baseline Model: Logistic Regression

Logistic Regression is used as a simple baseline model.

In [35]:
# Initialize scaler
scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train)

# Transform training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Initialize Logistic Regression
baseline_model = LogisticRegression(
    max_iter=3000,     # Increased iterations
    solver="lbfgs"
)

# Train model
baseline_model.fit(X_train_scaled, y_train)

# Predict
y_pred_baseline = baseline_model.predict(X_test_scaled)


### Evaluate Baseline Model

## Step 8 — Advanced Model: Random Forest

Random Forest can capture nonlinear relationships and interactions between features.

In [36]:

print("Confusion Matrix (Baseline):")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nClassification Report (Baseline):")
print(classification_report(y_test, y_pred_baseline))



Confusion Matrix (Baseline):
[[   0 1023    0]
 [   0 2480    0]
 [   0 1497    0]]

Classification Report (Baseline):
              precision    recall  f1-score   support

        High       0.00      0.00      0.00      1023
         Low       0.50      1.00      0.66      2480
      Medium       0.00      0.00      0.00      1497

    accuracy                           0.50      5000
   macro avg       0.17      0.33      0.22      5000
weighted avg       0.25      0.50      0.33      5000

/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted 

In [37]:


rf_model_balanced = RandomForestClassifier(
    n_estimators=200,                 # Slightly increased trees
    random_state=42,
    class_weight="balanced_subsample" # Handle imbalance
)

# Train model
rf_model_balanced.fit(X_train, y_train)

# Predict
y_pred_rf_balanced = rf_model_balanced.predict(X_test)

### Evaluate Random Forest Model

In [38]:
print("Confusion Matrix (Balanced Random Forest):")
print(confusion_matrix(y_test, y_pred_rf_balanced))

print("\nClassification Report (Balanced Random Forest):")
print(classification_report(y_test, y_pred_rf_balanced))

Confusion Matrix (Balanced Random Forest):
[[   5  912  106]
 [  16 2244  220]
 [  12 1320  165]]

Classification Report (Balanced Random Forest):
              precision    recall  f1-score   support

        High       0.15      0.00      0.01      1023
         Low       0.50      0.90      0.65      2480
      Medium       0.34      0.11      0.17      1497

    accuracy                           0.48      5000
   macro avg       0.33      0.34      0.27      5000
weighted avg       0.38      0.48      0.37      5000



## Step 9 — Save Best Model

We save the Random Forest model for deployment in Phase 5.

In [39]:

joblib.dump(rf_model, "risk_model.pkl")
print("Risk model saved successfully as risk_model.pkl")

Risk model saved successfully as risk_model.pkl


## Conclusion

- Logistic Regression provides a baseline comparison.
- Random Forest typically performs better for complex healthcare datasets.
- The model is now ready for evaluation and deployment preparation in later phases.